# P4 — Task 4 — Segmentation + dataset nộp kèm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanh-tai-435/BI10GroupYAPPERS/blob/main/notebooks/task4_segmentation.ipynb)

**Slide sở hữu:** 14, 15 · **Slide mới:** +1: Kiểm định & dịch chuyển segment  
**Đã có:** task4.py (k-means k=4, silhouette 0.251), task4_features.csv, chart t4_segment_sizes / t4_segment_profiles / t4_silhouette

**3 luật:** (1) Wellbeing, KHÔNG credit scoring — FHS không bao giờ dùng để từ chối/giảm hạn mức/khoá. (2) So bằng tỷ lệ, không so VND tuyệt đối. (3) Mỗi nhận định có số hoặc chart.

Code đầy đủ: `draft/task4_deep.py` (chạy độc lập được). Hướng dẫn chi tiết: `draft/YAPPERS_huong_dan_Colab.xlsx`, sheet của bạn.

In [ ]:
#@title ⚙️ SETUP — chạy 1 lần mỗi phiên Colab (không cần pip install gì cả)
# Colab có sẵn pandas / numpy / scikit-learn / matplotlib / openpyxl -> đủ cho mọi script trong draft/.
import os, re, sys, gzip, shutil, urllib.request
REPO = "/content/BI10GroupYAPPERS"
# Data tải thẳng từ thư mục Drive public (anyone with link) — không cần mount Drive / Add shortcut
# Thư mục: https://drive.google.com/drive/folders/1gOqCyY9IfFUJtbr2r3vR_Z3t61O71CrC
URL = {
    "consumer_financial_health_engagement_2025.csv": "https://drive.google.com/file/d/1YyueAH9Kasid6V_obml9pCMwh2dB0ryn/view",
    "consumer_transactions_2025.parquet.gz":         "https://drive.google.com/file/d/1O33KpFbUx6_BRGwFTb80xVSB3tzXqt5H/view",
    "data_dictionary.xlsx":                          "https://drive.google.com/file/d/1nFlZHmuaoK8VQ5xqRN2f4ZLqoR3gk2j1/view",
}

from google.colab import userdata

if os.path.exists(REPO):
    os.system(f"git -C {REPO} pull -q")
else:
    try:
        tok = userdata.get("GH_TOKEN")   # 🔑 Secrets bên trái: tên GH_TOKEN, bật "Notebook access"
    except Exception:                    # chưa tạo secret -> hỏi trực tiếp (không lưu vào notebook)
        from getpass import getpass
        tok = getpass("Chưa có secret GH_TOKEN — dán GitHub token (ghp_...) rồi Enter: ").strip()
    os.system(f"git clone -q https://{tok}@github.com/thanh-tai-435/BI10GroupYAPPERS.git {REPO}")
assert os.path.exists(REPO), "Clone thất bại: kiểm tra GH_TOKEN + đã được mời vào repo chưa"
os.chdir(REPO)

# Dựng lại đúng đường dẫn mà các script draft/*.py đang dùng
D = "BI10_ROUND01_DATASET"
TX = f"{D}/consumer_transactions_2025.parquet/consumer_transactions_2025.csv"
os.makedirs(os.path.dirname(TX), exist_ok=True)
for f, u in URL.items():
    if not os.path.exists(f"{D}/{f}"):
        fid = re.search(r"/d/([\w-]+)", u).group(1)   # tải file public bằng stdlib, không cần gdown
        urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", f"{D}/{f}")
if not os.path.exists(TX):   # .parquet.gz thực chất là CSV giao dịch nén gzip (134MB -> 597MB, ~30s)
    with gzip.open(f"{D}/consumer_transactions_2025.parquet.gz") as src, open(TX, "wb") as dst:
        shutil.copyfileobj(src, dst)

# Script gốc gọi sys.stdout.reconfigure(); stdout của Colab không có hàm này -> cho thành no-op
if not hasattr(sys.stdout, "reconfigure"):
    type(sys.stdout).reconfigure = lambda self, **k: None
os.makedirs("outputs/figures", exist_ok=True)
if not os.path.exists("draft/task4_features.csv"):   # nhãn segment (git-ignore) — task3/task5/charts cần
    os.system("python draft/task4.py > /dev/null")
print("✅ Sẵn sàng — cwd:", os.getcwd())


## Bước 0 — Chạy baseline
Nạp toàn bộ biến của script gốc (`mon`, `cons`, `txn`…) vào notebook. So kết quả với `draft/task4_output.txt`.

In [ ]:
%run -i draft/task4.py

## Việc 1 — Preprocessed Dataset (deliverable BẮT BUỘC)  `[BẮT BUỘC · hạn 27/09 12:00]`
Xuất task4_features.csv: 1 dòng/khách, 8 feature thô + 8 feature z-scale + nhãn segment + cột nhân khẩu. Kèm data_dictionary_task4.md (tên cột, ý nghĩa, công thức, đơn vị). Tải về từ Colab (thanh file bên trái) gửi P5 cho ZIP.

**Giao:** CSV + dictionary → P5

In [ ]:
# Việc 1 — Preprocessed Dataset: 8 feature thô + 8 feature z-scale + nhãn + nhân khẩu
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(cons[feat])            # cùng scaler với task4.py (fit trên 999 khách)
Z = pd.DataFrame(scaler.transform(cons[feat]), index=cons.index, columns=[f"z_{c}" for c in feat])
ds = cons[feat].join(Z).join(cons[["cluster", "segment", "rule2x2"]]).join(demo)
assert len(ds) == 999 and ds.segment.notna().all() and not ds.isnull().any().any()
ds.to_csv("draft/task4_features.csv")
print(ds.shape, "->", "draft/task4_features.csv")

DICT = {
    "consumer_id": "Mã khách (khoá, nối với 2 file gốc)",
    **{c: f"TB 12 tháng của `{c}` (file consumer-month), trục {FEATURES[c]}" for c in feat},
    **{f"z_{c}": f"`{c}` đã chuẩn hoá z-score (StandardScaler, fit trên 999 khách) — input của k-means" for c in feat},
    "cluster": "Nhãn cụm k-means (k=4, random_state=42, n_init=25)",
    "segment": "Tên segment nghiệp vụ gán theo cực trị của cụm",
    "rule2x2": "Nhãn kiểm chứng 2x2 health x engagement (chia theo median) — không phải mô hình chính",
    "age": "Tuổi (dùng để mô tả, KHÔNG đưa vào phân cụm)", "gender": "Giới tính (mô tả)",
    "occupation": "Nghề (mô tả)", "province": "Tỉnh/thành (mô tả)",
}
with open("draft/data_dictionary_task4.md", "w", encoding="utf-8") as f:
    f.write("# Data dictionary — task4_features.csv\n\n1 dòng / khách (999). Nguồn: consumer_financial_health_engagement_2025.csv, "
            "gộp 12 tháng bằng trung bình. Chỉ dùng tỷ lệ/điểm, không dùng VND tuyệt đối.\n\n| Cột | Ý nghĩa |\n|---|---|\n")
    f.writelines(f"| `{k}` | {v} |\n" for k, v in DICT.items())
print("-> draft/data_dictionary_task4.md")


## Việc 2 — Kiểm định độ bền (SLIDE MỚI)  `[Cao · hạn 27/09 12:00]`
(a) 20 random_state → ARI so nghiệm gốc (>0.8 = ổn định). (b) Bootstrap 80% × 50 lần → % khách giữ segment. (c) GMM k=4: ARI với k-means + % khách có xác suất < 0.6 ('khách biên'). (d) PCA 2D scatter tô màu segment.

**Giao:** Chart t4_pca.png + bảng ARI, slide mới

In [ ]:
# Việc 2a — Độ ổn định theo seed: 20 random_state, ARI so với nghiệm gốc (>0.8 = ổn định)
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score as ari
base = cons["cluster"].values
seed_ari = [ari(base, KMeans(K, n_init=10, random_state=s).fit_predict(X)) for s in range(20)]
print(f"ARI theo 20 seed: min {min(seed_ari):.3f} · TB {np.mean(seed_ari):.3f}")


In [ ]:
# Việc 2b — Bootstrap 80% x 50 lần: fit lại trên mẫu con, gán lại cả 999 khách, đo ARI
rng = np.random.default_rng(42)
boot_ari = []
for _ in range(50):
    idx = rng.choice(len(X), int(0.8 * len(X)), replace=False)
    boot_ari.append(ari(base, KMeans(K, n_init=10, random_state=42).fit(X[idx]).predict(X)))
print(f"ARI bootstrap 80%: TB {np.mean(boot_ari):.3f} · 5th pct {np.percentile(boot_ari, 5):.3f}")


In [ ]:
# Việc 2c — So với GMM (gán mềm): ARI k-means vs GMM + % khách 'ở biên' (xác suất cao nhất < 0.6)
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(K, covariance_type="full", n_init=5, random_state=42).fit(X)
g_lab, g_p = gmm.predict(X), gmm.predict_proba(X).max(1)
print(f"ARI k-means vs GMM: {ari(base, g_lab):.3f}")
print(f"Khách ở biên (p<0.6): {(g_p < 0.6).sum()} ({(g_p < 0.6).mean():.1%})")
print(pd.crosstab(cons["segment"], g_lab, rownames=["k-means"], colnames=["GMM"]))


In [ ]:
# Việc 2d — PCA 2D tô màu segment (chart cho slide mới)
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
pca = PCA(2, random_state=42).fit(X); P2 = pca.transform(X)
fig, ax = plt.subplots(figsize=(9, 6))
for s in cons["segment"].unique():
    m = (cons["segment"] == s).values
    ax.scatter(P2[m, 0], P2[m, 1], s=12, alpha=.7, label=f"{s} ({m.sum()})")
ax.set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)",
       ylabel=f"PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)",
       title="Four segments in PCA space (8 z-scaled features, 999 consumers)")
ax.legend(fontsize=8, loc="best"); fig.tight_layout()
fig.savefig("outputs/figures/t4_pca.png", dpi=150); plt.show()


## Việc 3 — Dịch chuyển segment theo tháng  `[Cao · hạn 27/09 18:00]`
Gán mỗi dòng consumer-month vào centroid gần nhất (cùng scaler) → ma trận chuyển tháng t → t+1. Bao nhiêu % Healthy trượt sang Stretched trong T11–T12? → bằng chứng 'stress theo đợt' + Planning Reminder trước T12 (gửi P5).

**Giao:** Ma trận chuyển + 1 con số gửi P5

In [ ]:
# Việc 3 — Dịch chuyển segment theo tháng: gán MỖI consumer-month vào centroid gần nhất (cùng scaler)
mon = mon.sort_values(["consumer_id", "analysis_month"])
name = dict(zip(cons["cluster"], cons["segment"]))
mon["seg"] = pd.Series(km.predict(scaler.transform(mon[feat])), index=mon.index).map(name)
mon["seg_next"] = mon.groupby("consumer_id")["seg"].shift(-1)
tr = mon.dropna(subset=["seg_next"])
print("Ma trận chuyển tháng t -> t+1 (% theo hàng):")
print((100 * pd.crosstab(tr.seg, tr.seg_next, normalize="index")).round(1))
H, S = "Financially Healthy & Highly Engaged", "Financially Stretched but Highly Engaged"
mon["m"] = pd.to_datetime(mon["analysis_month"]).dt.month
h = tr.assign(m=mon.loc[tr.index, "m"]).query("seg == @H")
slide = h.groupby("m")["seg_next"].apply(lambda s: (s == S).mean() * 100).round(1)
print("\n% tháng Healthy trượt sang Stretched ở tháng kế tiếp, theo tháng xuất phát:")
print(slide.to_string())
print(f"-> TB cả năm {slide.mean():.1f}% | T10->T11 {slide.get(10, np.nan)}% | T11->T12 {slide.get(11, np.nan)}%")


## Việc 4 — Hồ sơ nhân khẩu từng segment  `[Trung bình · hạn 27/09 18:00]`
Tuổi, giới, top tỉnh theo segment → chứng minh segment không trùng nhân khẩu → công bằng. Đồng bộ số giới tính (56.6% vs 49.4%) với P5.

**Giao:** Bảng slide 15

In [ ]:
# Việc 4 — Hồ sơ nhân khẩu từng segment (kiểm tra segment có trùng nhân khẩu học không)
bins = [14, 17, 24, 34, 44, 54, 64, 120]
labs = ["15-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]   # giữ 7 khách 15–17 tuổi
ds["age_grp"] = pd.cut(ds.age, bins, labels=labs)
print("Tuổi TB:\n", ds.groupby("segment").age.agg(["mean", "median"]).round(1))
print("\n% giới tính:\n", (100 * pd.crosstab(ds.segment, ds.gender, normalize="index")).round(1))
print("\n% nhóm tuổi:\n", (100 * pd.crosstab(ds.segment, ds.age_grp, normalize="index")).round(1))
print("\nTop 3 tỉnh mỗi segment:")
for s, g in ds.groupby("segment"):
    print(f"  {s}: " + ", ".join(f"{p} {v:.0%}" for p, v in g.province.value_counts(normalize=True).head(3).items()))
from scipy.stats import chi2_contingency
for col in ["gender", "age_grp"]:
    p = chi2_contingency(pd.crosstab(ds.segment, ds[col]))[1]
    print(f"Chi-square segment x {col}: p = {p:.3f}  ({'có liên quan' if p < .05 else 'không khác biệt có ý nghĩa'})")


## Việc 5 — Ánh xạ sang tên gợi ý của đề  `[Trung bình · hạn 27/09 18:00]`
Bảng 1 dòng / segment: vì sao 'Healthy but Disengaged' và 'Essential-Spend-Focused' không thành cụm riêng (nhóm 25 khách của P3 nằm ở segment nào?).

**Giao:** Bảng slide 14

In [ ]:
# Việc 5 — Ánh xạ sang 6 tên gợi ý của đề: 2x2 rule + 2 nhóm đề gợi ý nằm ở đâu
print((pd.crosstab(cons["segment"], cons["rule2x2"], margins=True)))
ess = cons["discretionary_spend_ratio"] < 0.40            # 'Essential-Spend-Focused': essential > 60%
print(f"\nEssential-focused (discretionary < 40%): {ess.sum()} khách")
print(cons.loc[ess, "segment"].value_counts())
hd = (cons.financial_health_score >= 70) & (cons.engagement_score < 70)   # cutoff của P3 (kiểm lại với P3)
print(f"\nHealthy but Disengaged (FHS>=70 & engagement<70): {hd.sum()} khách")
print(cons.loc[hd, "segment"].value_counts())


## Việc 6 — Limitations & Future Work  `[Thấp · hạn 27/09 18:00]`
≥3 bullet trên slide 15 hoặc slide mới.

**Giao:** Bullet

Gợi ý bullet (sửa lại bằng lời của bạn):
- **Kỹ thuật:** silhouette 0.251 (ranh giới mềm); gộp trung bình năm che mất quỹ đạo; ARI k-means vs GMM chỉ 0.26 → 3 cụm lớn là một dải liên tục, riêng Emerging Digital (88) trùng khớp hoàn toàn.
- **Dữ liệu:** dữ liệu tổng hợp 1 năm; engagement lệch cao; segment 88 khách kém bền nhất.
- **Công bằng:** segment có liên quan tuổi/giới (chi-square p<0.01) → can thiệp kích hoạt theo HÀNH VI, không nhắm theo tuổi/giới.
- **Future:** dữ liệu nhiều năm, đặc trưng xu hướng (độ dốc FHS), cảnh báo sớm trước tháng 12.

## Findings — kết quả đã chạy 27/09 (chi tiết: `draft/task4_deep_output.txt`)
- Ổn định theo seed: ARI TB **0.988** (min 0.931) · bootstrap 80%: ARI TB **0.911** → nghiệm k=4 ổn định.
- GMM: ARI với k-means **0.259**; chỉ **3.4%** khách ở biên. Emerging Digital (88) trùng 100%; 3 segment lớn là một dải liên tục (thấy rõ trên PCA) → nói thẳng trong Limitations.
- Dịch chuyển: tháng Healthy trượt sang Stretched tháng sau TB **23.3%**, nhưng **T11→T12 = 72.4%** → bằng chứng cho Planning Reminder trước tháng 12.
- Nhân khẩu: Power Users trẻ hơn (43.2 tuổi vs 50.6–57.9) và 60.5% nữ; chi-square p<0.01 → segment CÓ tương quan nhân khẩu dù không phân cụm theo nhân khẩu → kích hoạt theo hành vi.
- Tên gợi ý của đề: Essential-Spend-Focused chỉ **4** khách → không thành cụm; Healthy-but-Disengaged **25** khách, 24 nằm trong Emerging Digital.

---
**Lưu bài:** `File → Save a copy in GitHub` → repo `thanh-tai-435/BI10GroupYAPPERS`, branch `main`, path `notebooks/task4_segmentation.ipynb` (mỗi người chỉ lưu notebook của mình → không conflict). Chart: `savefig('outputs/figures/tN_ten.png')` rồi tải về gửi P5.